<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z344_MundoIdealEscalado2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mundo Ideal Escalado 2 — nuevas normalizaciones y Pearson

Extensión de z343. Agrega:

**Fede** → `diff_std`: diferencia con el mes anterior dividida por el desvío histórico
```
d[t] = (tn[t] - tn[t-1]) / std(serie)
```
Captura cambios relativos en vez de niveles. Invariante a escala y a nivel absoluto.

**El Chino (José Keh)** → Pearson como métrica de similitud
```
si f(x) = a·g(x) + b  →  pearson(f, g) = 1.0  siempre
```
Pearson es invariante a escala multiplicativa Y a traslación vertical. No necesita normalizar antes — detecta la misma forma aunque una serie sea 200x más grande o esté corrida hacia arriba.
También probamos Pearson sobre la serie diferenciada para remover tendencias.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial.distance import cosine
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.stats import pearsonr
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
T         = 36
K         = 12
HORIZONTE = 2
print('OK')

# 1. Generar dataset sintético (igual que z343)

In [ ]:
t = np.arange(T)

SHAPES = {
    'tendencia+': lambda: 1 + t/T*3 + np.random.normal(0, 0.05, T),
    'tendencia-': lambda: 4 - t/T*3 + np.random.normal(0, 0.05, T),
    'estable':    lambda: np.ones(T)*2 + np.random.normal(0, 0.08, T),
    'estacional': lambda: 1.5 + np.sin(2*np.pi*t/12) + 0.5*np.sin(2*np.pi*t/6) + np.random.normal(0, 0.05, T),
    'spike':      lambda: np.array([1.2]*T) + np.array([8.0 if i==T//3 else (5.0 if i==2*T//3 else 0) for i in range(T)]) + np.random.normal(0, 0.05, T),
    'V':          lambda: np.concatenate([np.linspace(3,0.5,T//2), np.linspace(0.5,3,T-T//2)]) + np.random.normal(0, 0.05, T),
    'escalon':    lambda: np.array([1.0 if i < T//2 else 4.0 for i in range(T)]) + np.random.normal(0, 0.05, T),
    'ruido':      lambda: np.abs(np.random.normal(2, 1.5, T)),
}
N_SHAPES     = len(SHAPES)
shape_nombres = list(SHAPES.keys())

def agregar_perturbaciones(serie, prob_inicio_tardio=0.4, prob_cero=0.1, rng=None):
    if rng is None: rng = np.random.default_rng()
    s = serie.copy()
    primer_mes = 0
    if rng.random() < prob_inicio_tardio:
        inicio = rng.integers(1, T//3)
        s[:inicio] = -1.0
        primer_mes = int(inicio)
    for j in range(primer_mes, T):
        if rng.random() < prob_cero:
            s[j] = 0.0
    return s, primer_mes

registros = []
rng = np.random.default_rng(99)

for shape_idx, (nombre_shape, fn) in enumerate(SHAPES.items()):
    for k in range(K):
        np.random.seed(shape_idx*100 + k)
        serie_base   = np.maximum(fn(), 0.0)
        escala_real  = float(np.exp(rng.uniform(np.log(0.05), np.log(200))))
        serie_escalada = serie_base * escala_real
        serie_final, primer_mes = agregar_perturbaciones(serie_escalada, rng=rng)
        registros.append({
            'id': f'{nombre_shape}_{k:02d}', 'shape': nombre_shape,
            'shape_idx': shape_idx, 'k': k,
            'escala_real': escala_real, 'primer_mes': primer_mes,
            'serie': serie_final, 'serie_base': serie_base,
        })

labels_reales = np.array([r['shape_idx'] for r in registros])
print(f'{len(registros)} series  |  {N_SHAPES} shapes × {K} variantes')

# 2. Normalizaciones — las 6 anteriores + diff_std (Fede)

In [ ]:
def preparar_vector(serie):
    v = serie.copy(); v[v == -1] = 0.0; return v

def norm_sin(serie):      return serie.copy(), 1.0

def norm_max(serie):
    reales = serie[serie >= 0]
    m = float(reales.max()) if len(reales) > 0 and reales.max() > 0 else 1.0
    n = serie.copy(); n[serie >= 0] = serie[serie >= 0] / m
    return n, m

def norm_l2(serie):
    reales = serie[serie >= 0]
    norma  = float(np.sqrt((reales**2).sum())) or 1.0
    n = serie.copy(); n[serie >= 0] = serie[serie >= 0] / norma
    return n, norma

def norm_index(serie, n_base=3):
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) >= 1 else 1.0
    if base == 0: base = 1.0
    n = serie.copy(); n[serie >= 0] = serie[serie >= 0] / base
    return n, base

def norm_reciente(serie, ventana=3):
    reales = serie[serie >= 0]
    ultimos = reales[-ventana:] if len(reales) >= 1 else np.array([1.0])
    base = float(ultimos.mean()) if ultimos.mean() > 0 else 1.0
    n = serie.copy(); n[serie >= 0] = serie[serie >= 0] / base
    return n, base

def norm_std(serie):
    reales = serie[serie >= 0]
    mu    = float(reales.mean()) if len(reales) > 0 else 0.0
    sigma = float(reales.std())  if len(reales) > 1 else 1.0
    if sigma == 0: sigma = 1.0
    n = serie.copy(); n[serie >= 0] = (serie[serie >= 0] - mu) / sigma
    mn = n[n >= -0.5].min() if any(n >= -0.5) else 0.0
    if mn < 0: n[serie >= 0] -= mn
    return n, (mu, sigma)

def norm_diff_std(serie):
    """
    Fede: diferencia con el mes anterior / desvío histórico.
    d[t] = (tn[t] - tn[t-1]) / std(serie_reales)
    - Los -1 (no existía) se tratan como NaN: no generan diferencia.
    - Escala guardada = (ultimo_valor_real, std) para poder desnormalizar.
    - La serie resultante tiene un elemento menos (no hay diferencia para t=0).
      Rellenamos t=0 con 0 para mantener el largo T.
    """
    reales = serie[serie >= 0]
    sigma  = float(reales.std()) if len(reales) > 1 else 1.0
    if sigma == 0: sigma = 1.0
    ultimo_real = float(reales[-1]) if len(reales) > 0 else 0.0

    n = np.zeros(T)
    for i in range(1, T):
        if serie[i] >= 0 and serie[i-1] >= 0:
            n[i] = (serie[i] - serie[i-1]) / sigma
        elif serie[i] == -1 or serie[i-1] == -1:
            n[i] = -1.0   # sentinela: no existía
        else:
            n[i] = 0.0
    # t=0: no hay mes anterior → 0
    n[0] = 0.0 if serie[0] >= 0 else -1.0

    return n, (ultimo_real, sigma)


NORMALIZACIONES = {
    'sin_norm':  norm_sin,
    'max':       norm_max,
    'l2':        norm_l2,
    'index':     norm_index,
    'reciente':  norm_reciente,
    'std':       norm_std,
    'diff_std':  norm_diff_std,
}
print('Normalizaciones:', list(NORMALIZACIONES.keys()))

In [ ]:
# Visualizar diff_std en 4 variantes del shape estacional
nombre_test = 'estacional'
variantes_test = [r for r in registros if r['shape'] == nombre_test][:4]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for r in variantes_test:
    s = np.where(r['serie'] == -1, np.nan, r['serie'])
    ax.plot(s, alpha=0.8, linewidth=1.5)
ax.set_title('Original — 4 variantes estacional (distintas escalas)', fontsize=10)
ax.grid(alpha=0.3)

ax = axes[1]
for r in variantes_test:
    s_norm, _ = norm_diff_std(r['serie'])
    s = np.where(s_norm == -1, np.nan, s_norm)
    ax.plot(s, alpha=0.8, linewidth=1.5)
ax.set_title('diff_std — si funciona, las 4 líneas deberían superponerse', fontsize=10)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.4)
ax.grid(alpha=0.3)

plt.suptitle('Efecto de diff_std sobre el shape estacional', fontsize=11)
plt.tight_layout()
plt.show()

# 3. El Chino — Pearson como métrica de similitud

Pearson mide correlación lineal entre dos series. Es invariante a escala multiplicativa **y** a traslación vertical:
```
si g(x) = a·f(x) + b  →  pearson(f, g) = 1.0
```
No necesita normalizar — detecta la misma forma aunque una serie sea 200x más grande o esté corrida.

Lo usamos como **distancia**: `dist_pearson = 1 - |pearson|`  (0 = idénticas, 1 = sin relación)

In [ ]:
def pearson_dist(v1, v2):
    """Distancia basada en Pearson: 1 - |r|. Rango [0, 1]."""
    if v1.std() == 0 or v2.std() == 0:
        return 1.0
    r, _ = pearsonr(v1, v2)
    return 1.0 - abs(r)


def calcular_distancias_coseno(registros, norm_fn):
    vectores, labels = [], []
    for r in registros:
        v_norm, _ = norm_fn(r['serie'])
        vectores.append(preparar_vector(v_norm))
        labels.append(r['shape_idx'])
    n = len(vectores)
    intra, inter = [], []
    for i in range(n):
        for j in range(i+1, n):
            d = cosine(vectores[i], vectores[j])
            (intra if labels[i]==labels[j] else inter).append(d)
    return np.array(intra), np.array(inter)


def calcular_distancias_pearson(registros, diferenciada=False):
    """Pearson sobre la serie cruda o diferenciada."""
    vectores, labels = [], []
    for r in registros:
        v = preparar_vector(r['serie'])
        if diferenciada:
            v = np.diff(v, prepend=v[0])  # diferencia simple
        vectores.append(v)
        labels.append(r['shape_idx'])
    n = len(vectores)
    intra, inter = [], []
    for i in range(n):
        for j in range(i+1, n):
            d = pearson_dist(vectores[i], vectores[j])
            (intra if labels[i]==labels[j] else inter).append(d)
    return np.array(intra), np.array(inter)


print('OK')

In [ ]:
# Calcular todas las distancias
print('Calculando distancias...')
print()

resultados = {}

# coseno por normalización
for nombre_norm, fn in NORMALIZACIONES.items():
    intra, inter = calcular_distancias_coseno(registros, fn)
    sep = inter.mean() - intra.mean()
    resultados[f'coseno_{nombre_norm}'] = {'intra': intra, 'inter': inter, 'sep': sep}
    print(f'coseno_{nombre_norm:12s}  intra={intra.mean():.4f}  inter={inter.mean():.4f}  sep={sep:.4f}')

print()

# pearson crudo
intra, inter = calcular_distancias_pearson(registros, diferenciada=False)
sep = inter.mean() - intra.mean()
resultados['pearson_crudo'] = {'intra': intra, 'inter': inter, 'sep': sep}
print(f'pearson_crudo          intra={intra.mean():.4f}  inter={inter.mean():.4f}  sep={sep:.4f}')

# pearson diferenciado
intra, inter = calcular_distancias_pearson(registros, diferenciada=True)
sep = inter.mean() - intra.mean()
resultados['pearson_diff'] = {'intra': intra, 'inter': inter, 'sep': sep}
print(f'pearson_diff           intra={intra.mean():.4f}  inter={inter.mean():.4f}  sep={sep:.4f}')

In [ ]:
# Histogramas — todas las métricas
n_metricas = len(resultados)
ncols = 3
nrows = (n_metricas + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, (nombre, res) in enumerate(resultados.items()):
    ax = axes[i]
    ax.hist(res['intra'], bins=40, alpha=0.6, color='steelblue', label='intra (mismo shape)')
    ax.hist(res['inter'], bins=40, alpha=0.5, color='tomato',    label='inter (distinto shape)')
    ax.set_title(f'{nombre}  |  sep={res["sep"]:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

for j in range(i+1, len(axes)): axes[j].set_visible(False)

fig.suptitle('Distribución de distancias — coseno por normalización + Pearson (crudo y diferenciado)', fontsize=11)
plt.tight_layout()
plt.show()

# 4. Heatmap de correlaciones de Pearson — El Chino

In [ ]:
# Matriz de correlaciones Pearson entre las 96 series
# Si funciona bien: bloques diagonales = alta correlación (mismo shape)

n = len(registros)
# ordenar por shape para que los bloques queden juntos
registros_ord = sorted(registros, key=lambda r: r['shape_idx'])
labels_ord    = [r['shape_idx'] for r in registros_ord]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, diferenciada, titulo in [
    (axes[0], False, 'Pearson — serie cruda'),
    (axes[1], True,  'Pearson — serie diferenciada'),
]:
    mat = np.zeros((n, n))
    for i, r1 in enumerate(registros_ord):
        for j, r2 in enumerate(registros_ord):
            v1 = preparar_vector(r1['serie'])
            v2 = preparar_vector(r2['serie'])
            if diferenciada:
                v1 = np.diff(v1, prepend=v1[0])
                v2 = np.diff(v2, prepend=v2[0])
            if v1.std() > 0 and v2.std() > 0:
                r_val, _ = pearsonr(v1, v2)
                mat[i, j] = abs(r_val)
            else:
                mat[i, j] = 0.0

    im = ax.imshow(mat, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, fraction=0.046)

    # líneas divisorias entre shapes
    for s in range(1, N_SHAPES):
        pos = s * K - 0.5
        ax.axhline(pos, color='black', linewidth=1.2, alpha=0.7)
        ax.axvline(pos, color='black', linewidth=1.2, alpha=0.7)

    # etiquetas de shapes
    tick_pos = [s * K + K//2 for s in range(N_SHAPES)]
    ax.set_xticks(tick_pos); ax.set_xticklabels(shape_nombres, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(tick_pos); ax.set_yticklabels(shape_nombres, fontsize=8)
    ax.set_title(titulo, fontsize=11)

fig.suptitle('Heatmap |Pearson| entre las 96 series (ordenadas por shape)\n'
             'Bloques diagonales brillantes = mismo shape bien identificado', fontsize=11)
plt.tight_layout()
plt.show()
print('Bloques diagonales brillantes = el shape se reconoce bien con Pearson.')
print('Bloques off-diagonal brillantes = shapes que se confunden entre sí.')

# 5. Clustering con Pearson como distancia

In [ ]:
def ari_clustering_coseno(registros, norm_fn):
    vectores = [preparar_vector(norm_fn(r['serie'])[0]) for r in registros]
    Z = linkage(vectores, method='ward')
    labels_pred = fcluster(Z, N_SHAPES, criterion='maxclust')
    return adjusted_rand_score(labels_reales, labels_pred), Z


def ari_clustering_pearson(registros, diferenciada=False):
    n = len(registros)
    # matriz de distancias condensada para linkage
    dist_condensada = []
    for i in range(n):
        for j in range(i+1, n):
            v1 = preparar_vector(registros[i]['serie'])
            v2 = preparar_vector(registros[j]['serie'])
            if diferenciada:
                v1 = np.diff(v1, prepend=v1[0])
                v2 = np.diff(v2, prepend=v2[0])
            dist_condensada.append(pearson_dist(v1, v2))
    Z = linkage(dist_condensada, method='average')  # ward requiere euclidiana, usamos average
    labels_pred = fcluster(Z, N_SHAPES, criterion='maxclust')
    return adjusted_rand_score(labels_reales, labels_pred), Z


print('ARI por método:')
print()

ari_resultados = {}

for nombre_norm, fn in NORMALIZACIONES.items():
    ari, Z = ari_clustering_coseno(registros, fn)
    ari_resultados[f'coseno_{nombre_norm}'] = (ari, Z)
    print(f'  coseno_{nombre_norm:12s}: ARI = {ari:.4f}')

print()
ari, Z = ari_clustering_pearson(registros, diferenciada=False)
ari_resultados['pearson_crudo'] = (ari, Z)
print(f'  pearson_crudo          : ARI = {ari:.4f}')

ari, Z = ari_clustering_pearson(registros, diferenciada=True)
ari_resultados['pearson_diff'] = (ari, Z)
print(f'  pearson_diff           : ARI = {ari:.4f}')

print()
mejor = max(ari_resultados, key=lambda k: ari_resultados[k][0])
print(f'  ★ Mejor: {mejor}  (ARI={ari_resultados[mejor][0]:.4f})')

In [ ]:
# Árbol de decisión — incluye diff_std y comparamos con Pearson features
print('Accuracy 5-fold CV árbol de decisión:')
print()

arbol_resultados = {}

# coseno normalizations → features = serie normalizada
for nombre_norm, fn in NORMALIZACIONES.items():
    X = np.array([preparar_vector(fn(r['serie'])[0]) for r in registros])
    clf = DecisionTreeClassifier(max_depth=5, random_state=42)
    scores = cross_val_score(clf, X, labels_reales, cv=5, scoring='accuracy')
    arbol_resultados[f'arbol_{nombre_norm}'] = scores
    print(f'  {nombre_norm:12s}: {scores.mean():.4f} ± {scores.std():.4f}')

print()
# Pearson features: vector de correlaciones con todas las demás series
# (alternativa: usar los valores crudos y dejar que el árbol use pearson implícitamente)
print('  [Pearson no aplica directamente al árbol — usa clustering separado]')

# 6. Visualización diff_std — ¿qué forma toma la serie?

In [ ]:
# Mostrar diff_std para todos los shapes (1 variante por shape)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, nombre_shape in enumerate(SHAPES):
    r = next(x for x in registros if x['shape'] == nombre_shape)
    ax = axes[i]

    s_orig = np.where(r['serie'] == -1, np.nan, r['serie'])
    s_norm, _ = norm_diff_std(r['serie'])
    s_diff = np.where(s_norm == -1, np.nan, s_norm)

    ax2 = ax.twinx()
    ax.plot(s_orig, color='steelblue', alpha=0.6, linewidth=1.5, label='original')
    ax2.plot(s_diff, color='tomato', alpha=0.8, linewidth=1.2, linestyle='--', label='diff_std')
    ax2.axhline(0, color='gray', linewidth=0.7, linestyle=':')

    ax.set_title(nombre_shape, fontsize=9)
    ax.set_ylabel('original', fontsize=7, color='steelblue')
    ax2.set_ylabel('diff_std', fontsize=7, color='tomato')
    ax.grid(alpha=0.2)

fig.suptitle('diff_std por shape — azul=original, rojo=diferencias/desvío', fontsize=11)
plt.tight_layout()
plt.show()
print('tendencia+/- → diff_std es constante positiva/negativa')
print('estacional   → diff_std oscila simétricamente alrededor de 0')
print('spike        → diff_std tiene dos picos enormes (subida y bajada del spike)')
print('escalon      → diff_std es 0 en ambos tramos + un pico en el momento del escalón')

# 7. Resumen comparativo final

In [ ]:
print('=' * 72)
print(f'  {"MÉTODO":28s}  {"Separación":>11s}  {"ARI Cluster":>11s}  {"Acc Árbol":>10s}')
print('=' * 72)

for key in ari_resultados:
    ari_val = ari_resultados[key][0]
    sep_val = resultados.get(key, {}).get('sep', '-')
    acc_key = key.replace('coseno_', 'arbol_')
    acc_val = arbol_resultados[acc_key].mean() if acc_key in arbol_resultados else '-'

    sep_str = f'{sep_val:.4f}' if isinstance(sep_val, float) else sep_val
    acc_str = f'{acc_val:.4f}' if isinstance(acc_val, float) else acc_val
    print(f'  {key:28s}  {sep_str:>11s}  {ari_val:>11.4f}  {acc_str:>10s}')

print('=' * 72)
print()
print('Separación: inter.mean - intra.mean en distancia coseno/pearson')
print('ARI:        1.0 = clustering perfecto, 0.0 = aleatorio')
print('Acc árbol:  fracción correcta en clasificación supervisada 5-fold')

In [ ]:
# Bar chart resumen ARI
nombres = list(ari_resultados.keys())
valores = [ari_resultados[k][0] for k in nombres]
colores = ['#2ecc71' if v > 0.5 else ('#f39c12' if v > 0.25 else '#e74c3c') for v in valores]

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(nombres, valores, color=colores, edgecolor='white', linewidth=1.2)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='umbral 0.5')
ax.set_xticklabels(nombres, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('ARI')
ax.set_title('ARI del clustering jerárquico — todas las métricas', fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.4)
for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()